In [11]:
import pandas as pd
import os

data_path ="../data/raw/"
files = [
    "olist_customers_dataset.csv", "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv", "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv", "olist_orders_dataset.csv",
    "olist_products_dataset.csv", "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
]
dfs={}
for f in files:
    name= f.replace("olist_","").replace("_dataset.csv","").replace(".csv","")
    dfs[name] = pd.read_csv(os.path.join(data_path,f))
    
for name, df in dfs.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) > 0:
        print(f"\n{name}:\n{nulls}")
    else:
        print(f"\n{name}: no missing values")
dfs["products"][dfs["products"]["product_category_name"].isnull()].shape[0]
dfs["products"][dfs["products"]["product_category_name"].isnull() & dfs["products"]["product_name_lenght"].isnull()].shape[0]


customers: no missing values

geolocation: no missing values

order_items: no missing values

order_payments: no missing values

order_reviews:
review_comment_title      87656
review_comment_message    58247
dtype: int64

orders:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

products:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

sellers: no missing values

product_category_name_translation: no missing values


610

In [12]:
for name, df in dfs.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")
print("Unique zip prefixes:", dfs["geolocation"]["geolocation_zip_code_prefix"].nunique())
print("Total rows:", dfs["geolocation"].shape[0])

customers: 0 duplicate rows
geolocation: 261831 duplicate rows
order_items: 0 duplicate rows
order_payments: 0 duplicate rows
order_reviews: 0 duplicate rows
orders: 0 duplicate rows
products: 0 duplicate rows
sellers: 0 duplicate rows
product_category_name_translation: 0 duplicate rows
Unique zip prefixes: 19015
Total rows: 1000163


In [10]:
for name, df in dfs.items():
    print(f"\n{name}:\n{df.dtypes}")


customers:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

geolocation:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

order_items:
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

order_payments:
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

order_reviews:
review_id                  object
order_id                   object
review_score                int64
review_comment_title  

In [13]:
print(dfs["orders"]["order_status"].value_counts())
print(dfs["order_payments"]["payment_type"].value_counts())
print(dfs["order_reviews"]["review_score"].value_counts())

print("Negative price:", (dfs["order_items"]["price"] < 0).sum())
print("Negative freight:", (dfs["order_items"]["freight_value"] < 0).sum())
print("Negative payment_value:", (dfs["order_payments"]["payment_value"] < 0).sum())
print("Zero payment_value:", (dfs["order_payments"]["payment_value"] == 0).sum())

orders_dt = dfs["orders"].copy()
orders_dt["order_purchase_timestamp"] = pd.to_datetime(orders_dt["order_purchase_timestamp"])
orders_dt["order_delivered_customer_date"] = pd.to_datetime(orders_dt["order_delivered_customer_date"])
bad_dates = orders_dt[orders_dt["order_delivered_customer_date"] < orders_dt["order_purchase_timestamp"]]
print("Orders delivered before purchase (invalid):", bad_dates.shape[0])

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64
review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64
Negative price: 0
Negative freight: 0
Negative payment_value: 0
Zero payment_value: 9
Orders delivered before purchase (invalid): 0


In [17]:
print(dfs["order_items"][["price", "freight_value"]].describe())
print(dfs["order_payments"][["payment_value", "payment_installments"]].describe())
print(dfs["products"][["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]].describe())

q1 = dfs["order_items"]["price"].quantile(0.25)
q3 = dfs["order_items"]["price"].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
outliers = dfs["order_items"][(dfs["order_items"]["price"] < lower) | (dfs["order_items"]["price"] > upper)]
print(f"\nPrice outliers (IQR method): {outliers.shape[0]}")



               price  freight_value
count  112650.000000  112650.000000
mean      120.653739      19.990320
std       183.633928      15.806405
min         0.850000       0.000000
25%        39.900000      13.080000
50%        74.990000      16.260000
75%       134.900000      21.150000
max      6735.000000     409.680000
       payment_value  payment_installments
count  103886.000000         103886.000000
mean      154.100380              2.853349
std       217.494064              2.687051
min         0.000000              0.000000
25%        56.790000              1.000000
50%       100.000000              1.000000
75%       171.837500              4.000000
max     13664.080000             24.000000
       product_weight_g  product_length_cm  product_height_cm  \
count      32949.000000       32949.000000       32949.000000   
mean        2276.472488          30.815078          16.937661   
std         4282.038731          16.914458          13.637554   
min            0.000000      

In [18]:
pk_map = {
    "customers": "customer_id",
    "orders": "order_id",
    "products": "product_id",
    "sellers": "seller_id",
    "order_reviews": "review_id",
    "product_category_name_translation": "product_category_name",
}
for name, key in pk_map.items():
    total = dfs[name].shape[0]
    unique = dfs[name][key].nunique()
    print(f"{name}: {key} → total={total}, unique={unique}, is_PK={total==unique}")

customers: customer_id → total=99441, unique=99441, is_PK=True
orders: order_id → total=99441, unique=99441, is_PK=True
products: product_id → total=32951, unique=32951, is_PK=True
sellers: seller_id → total=3095, unique=3095, is_PK=True
order_reviews: review_id → total=99224, unique=98410, is_PK=False
product_category_name_translation: product_category_name → total=71, unique=71, is_PK=True


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [16]:
composite_dupes = dfs["order_reviews"].duplicated(subset=["review_id", "order_id"]).sum()
print("Duplicate (review_id, order_id) pairs:", composite_dupes)

Duplicate (review_id, order_id) pairs: 0
